# PHASE 1 V2 — Chuẩn bị: phân đoạn + 3 nguồn xác định (KHÔNG LLM)

Kiến trúc V2 (xem `PIPELINE_V2.md`): LLM đọc **mọi** unit ở Phase 2, không
còn khái niệm "vùng mờ" như V1. Phase 1 chỉ làm phần không cần LLM:

- **K1** — phân đoạn văn bản thành unit (bullet/câu), gắn heading cha + zone
  (99% bullet có heading cha tự khai báo loại — xem NL2 trong `PIPELINE_V2.md`)
- **K3** — luật số+đơn vị (tất định, `extract_lab_pairs`, giữ nguyên từ V1)
- **K4** — từ điển RxNorm (`DrugMatcher`, giữ nguyên từ V1)
- **K5** — encoder + cascade KB ICD -> TRIỆU_CHỨNG/CHẨN_ĐOÁN (F1 0.82)

Ghi ra:
- `units/<id>.json` — danh sách unit, Phase 2 dùng để dựng prompt LLM
- `sources/<id>.json` — 3 nguồn CHƯA hợp nhất `{text, rule, dict, encoder}`;
  hợp nhất với LLM diễn ra ở Phase 2 qua `merge_entities.merge()`

Không còn file `todo_llm/` — Phase 2 không cần bảng vùng-mờ vì LLM đọc mọi
unit không điều kiện.

Settings: GPU T4 · Internet ON

In [ ]:
# Cell 1 — env tắt JIT (né nvrtc) TRƯỚC import torch, cài pyvi
import os
os.environ['PYTORCH_JIT'] = '0'                 # tắt TorchScript JIT
os.environ['PYTORCH_TENSOREXPR_FALLBACK'] = '2'
import sys, glob, json, re, subprocess, time
subprocess.run([sys.executable,'-m','pip','install','-q','pyvi'])
print('pyvi cài xong')

In [ ]:
# Cell 2 — dò model + code + KB + danh sách file
def find_dir(name):
    h = [x for x in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if os.path.isdir(x)]
    return h[0] if h else None

MODEL_DIR = find_dir('ner_encoder')
assert MODEL_DIR and os.path.exists(f'{MODEL_DIR}/model.safetensors'), 'thiếu ner_encoder'
if not os.path.exists('fakeer'):
    subprocess.run(['git','clone','-q','https://github.com/Khanhhh239/fakeer'])
ROOT = 'fakeer' if os.path.exists('fakeer/src') else os.path.dirname(os.path.dirname(find_dir('src') or ''))
sys.path.insert(0, f'{ROOT}/src')
def kb(n):
    p = glob.glob(f'{ROOT}/kb/{n}') or glob.glob(f'/kaggle/input/**/kb/{n}', recursive=True)
    assert p, f'thiếu KB {n}'; return p[0]
ICD, RXN, INN = kb('icd10_vi_full.csv'), kb('rxnorm_merged.csv'), kb('inn_usan.csv')

# thư mục 100 file .txt: ưu tiên input/ trong dataset
IN = find_dir('input') or (f'{ROOT}/input' if os.path.exists(f'{ROOT}/input') else None)
FILES = sorted(glob.glob(f'{IN}/*.txt'), key=lambda x: int(re.sub(r'\D','',os.path.basename(x)) or 0)) if IN else []
assert FILES, 'không thấy file .txt đầu vào'
os.makedirs('/kaggle/working/units', exist_ok=True)
os.makedirs('/kaggle/working/sources', exist_ok=True)
print(f'MODEL={MODEL_DIR}\nROOT={ROOT}\n{len(FILES)} file đầu vào')

In [ ]:
# Cell 3 — nạp encoder (thử GPU, fallback CPU) + cache embedding ICD ra .npy
import torch, numpy as np
for _f in ('_jit_set_texpr_fuser_enabled','_jit_set_nvfuser_enabled'):
    try: getattr(torch._C, _f)(False)
    except Exception: pass
try: torch._C._jit_override_can_fuse_on_gpu(False)
except Exception: pass

from transformers import AutoTokenizer, AutoModelForTokenClassification
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).eval()
ID2LAB = {int(k): v for k, v in mdl.config.id2label.items()}

# thử 1 forward trên GPU; nvrtc lỗi -> CPU
DEV = 'cpu'
if torch.cuda.is_available():
    try:
        mdl.to('cuda')
        _e = tok(['test'], is_split_into_words=True, return_tensors='pt', truncation=True, max_length=8).to('cuda')
        with torch.no_grad(): mdl(**_e)
        DEV = 'cuda'
    except Exception as _ex:
        print('encoder GPU lỗi -> CPU:', type(_ex).__name__, str(_ex)[:80]); mdl.to('cpu'); DEV = 'cpu'
print('encoder device:', DEV)

# --- ICD embedding: cache .npy để không encode lại 14.6k tên mỗi lần ---
import pandas as pd
_df = pd.read_csv(ICD)
_col = 'term' if 'term' in _df.columns else _df.columns[1]
ICD_CODES = _df['code'].astype(str).tolist()
ICD_NAMES = _df[_col].astype(str).tolist()
CACHE = '/kaggle/working/icd_emb.npy'
INP_CACHE = next(iter(glob.glob('/kaggle/input/**/icd_emb.npy', recursive=True)), None)

from transformers import AutoModel
EMB_NAME = 'AITeamVN/Vietnamese_Embedding'
etok = AutoTokenizer.from_pretrained(EMB_NAME)
emdl = AutoModel.from_pretrained(EMB_NAME).eval().to(DEV)
def embed(texts, bs=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            b = texts[i:i+bs]
            enc = etok(b, padding=True, truncation=True, max_length=64, return_tensors='pt').to(DEV)
            h = emdl(**enc).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1).float()
            e = (h*m).sum(1) / m.sum(1).clamp(min=1e-9)
            e = torch.nn.functional.normalize(e, p=2, dim=1)
            out.append(e.cpu().numpy())
    return np.vstack(out).astype('float32')

if INP_CACHE:
    ICD_EMB = np.load(INP_CACHE); print('nạp ICD embedding từ cache dataset')
else:
    t0 = time.time(); ICD_EMB = embed(ICD_NAMES); np.save(CACHE, ICD_EMB)
    print(f'encode {len(ICD_NAMES)} tên ICD trong {time.time()-t0:.0f}s -> lưu {CACHE} (dùng lại lần sau)')
R_MASK = np.array([c.startswith('R') for c in ICD_CODES])
print('ICD_EMB', ICD_EMB.shape)

In [ ]:
# Cell 4 — K1 (phân đoạn) + K3 (luật XN) + K4 (từ điển thuốc) + K5 (encoder/cascade)
from segment_units import segment_document
from branch_b_lab_tests import extract_lab_pairs
from branch_c_drugs import DrugMatcher
from utils.text_alignment import segment_with_map
DRUG = DrugMatcher(RXN, INN)
TAU_KB = 0.93   # tier2: cosine >= -> CHẨN_ĐOÁN chắc

def encoder_spans(text):
    words, spans, ok = segment_with_map(text)
    if not ok or not words: return []
    W, S = 120, 100; best_lab = [None]*len(words); best_c = [-1]*len(words); i = 0
    while i < len(words):
        chunk = words[i:i+W]
        enc = tok(chunk, is_split_into_words=True, return_tensors='pt', truncation=True, max_length=256).to(DEV)
        with torch.no_grad(): pred = mdl(**enc).logits.argmax(-1)[0].cpu().numpy()
        wids = enc.word_ids(0); prev = None
        for pos, wid in enumerate(wids):
            if wid is None or wid == prev: prev = wid; continue
            prev = wid; gi = i+wid; c = min(wid, len(chunk)-1-wid)
            if c > best_c[gi]: best_c[gi] = c; best_lab[gi] = ID2LAB[int(pred[pos])]
        if i+W >= len(words): break
        i += S
    out, a = [], None
    def close(b):
        if a is None: return
        s, e = spans[a][0], spans[b][1]; out.append({'text': text[s:e], 'start': s, 'end': e})
    for k, lab in enumerate((best_lab or [])+['O']):
        lab = lab or 'O'
        if lab == 'B-SYM_DIS':
            if a is not None: close(k-1)
            a = k
        elif lab == 'I-SYM_DIS':
            if a is None: a = k
        else:
            if a is not None: close(k-1); a = None
    return out

def cascade_kb(span_text):
    """tier1: khớp chương R -> TRIỆU_CHỨNG. tier2: ngoài R & cos>=TAU -> CHẨN_ĐOÁN.
       không quyết được -> None. K2 (LLM, Phase 2) sẽ tự đọc UNIT chứa vùng này
       và đề xuất type riêng — không cần đánh dấu 'todo' như V1 nữa, vì K2 đọc
       MỌI unit không điều kiện (xem PIPELINE_V2.md, bỏ khái niệm 'vùng mờ')."""
    q = embed([span_text])[0]; sims = ICD_EMB @ q
    r_top = sims[R_MASK].max() if R_MASK.any() else -1
    o_top = sims[~R_MASK].max() if (~R_MASK).any() else -1
    if r_top > 0.5 and r_top >= o_top: return 'TRIỆU_CHỨNG'
    if o_top >= TAU_KB: return 'CHẨN_ĐOÁN'
    return None
print('K1/K3/K4/K5 sẵn sàng')

In [ ]:
# Cell 5 — VÒNG LẶP 100 file: K1 phân đoạn + K3/K4/K5 nguồn xác định
t_all = time.time(); n_units = n_rule = n_dict = n_enc = 0
for fp in FILES:
    fid = os.path.splitext(os.path.basename(fp))[0]
    TEXT = open(fp, encoding='utf-8').read()

    units = segment_document(TEXT)

    rule_ents = extract_lab_pairs(TEXT)
    dict_ents = DRUG.extract_drugs(TEXT)

    encoder_ents = []
    for s in encoder_spans(TEXT):
        lab = cascade_kb(s['text'])
        if lab:
            encoder_ents.append({**s, 'type': lab, 'score': 0.9, 'source': 'encoder+kb'})

    # bất biến offset PHẢI đúng trước khi ghi — sai ở đây thì mọi thứ phía
    # sau (neo LLM, hợp nhất, xuất BTC) đều vô nghĩa. Chết ngay tại chỗ.
    for e in rule_ents + dict_ents + encoder_ents:
        assert TEXT[e['start']:e['end']] == e['text'], f'{fid}: {e}'
    for u in units:
        assert TEXT[u['start']:u['end']] == u['text'], f'{fid}: unit lệch offset {u}'

    json.dump(units, open(f'/kaggle/working/units/{fid}.json', 'w', encoding='utf-8'),
              ensure_ascii=False)
    json.dump({'text': TEXT, 'rule': rule_ents, 'dict': dict_ents, 'encoder': encoder_ents},
              open(f'/kaggle/working/sources/{fid}.json', 'w', encoding='utf-8'),
              ensure_ascii=False)

    n_units += len(units); n_rule += len(rule_ents); n_dict += len(dict_ents); n_enc += len(encoder_ents)

print(f'\nXONG {len(FILES)} file trong {time.time()-t_all:.0f}s')
print(f'{n_units} unit | rule {n_rule} | dict {n_dict} | encoder+kb {n_enc}')
print('=> /kaggle/working/units/*.json  +  /kaggle/working/sources/*.json  (+ icd_emb.npy)')